In [14]:
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou
from pathlib import Path
import torch
import cv2
import os
import shutil
import sys

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

True
0
NVIDIA GeForce RTX 4050 Laptop GPU


In [15]:
detect_model = YOLO("models/11s.pt")
pose_model = YOLO("yolo11s-pose")

In [16]:
video_input_directory = Path("video/input")
video_output_directory = Path("video/output")   

video_input_directory.mkdir(parents=True, exist_ok=True)
video_output_directory.mkdir(parents=True, exist_ok=True)

for item in video_output_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [17]:
for video_index, video in enumerate(video_input_directory.glob("*.mp4")):
    print(f"Video: {video}")

    detect_results = detect_model.track(
        device=0,
        source=video,
        tracker="botsort.yaml",
        persist=True,
        conf=0.5,
        stream=True,
        verbose=False,
        vid_stride=10,
    )

    cap = cv2.VideoCapture(str(video))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1
    cap.release()
    
    output_video = str(video_output_directory / f"video{video_index}.mp4")
    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    ioa_output_log = str(video_output_directory / f"ioa_log{video_index}.txt")
    hoi_output_log = str(video_output_directory / f"hoi_log{video_index}.txt")
    
    with open(ioa_output_log, "w") as ioa_log, open(hoi_output_log, "w") as hoi_log:
        try:
            for result_index, result in enumerate(detect_results):
                
                ioa_detected = False
                interaction_detected = False
                est_time_seconds = int(result_index / fps)
                minutes, seconds = divmod(est_time_seconds, 60)
                formatted_time = f"{minutes:02}:{seconds:02}"
                log_prefix = f"Frame {result_index:05} @ {formatted_time} >> "
                log_line = log_prefix
                log_line2 = log_prefix

                print(f"\rProcessing Frame: {result_index}/{total_frames}", end="", flush=True)

                frame = result.orig_img.copy()
                boxes = result.boxes
                
                if len(boxes) == 0:
                    writer.write(frame)
                    continue

                xyxys = boxes.xyxy.cpu().numpy()
                clss = boxes.cls.cpu().numpy().astype(int)
                confs = boxes.conf.cpu().numpy()
                track_ids = boxes.id.cpu().numpy().astype(int) if boxes.id is not None else [0] * len(boxes)
                
                person_indices = [i for i, c in enumerate(clss) if detect_model.names[c] == "person"]
                human_detected = len(person_indices) > 0

                pose_result = None
                kpts = None
                if human_detected:
                    pose_results = pose_model(
                        source=frame,
                        show_labels=False,
                        device=0,
                        conf=0.5,
                        verbose=False
                    )
                    pose_result = pose_results[0]
                    frame = pose_result.plot(boxes=False)
                    if pose_result.keypoints is not None:
                        kpts = pose_result.keypoints.xy.cpu().numpy()

                for i, (x1, y1, x2, y2) in enumerate(xyxys):
                    conf = round(float(confs[i]), 2)
                    cls_id = clss[i]
                    class_name = detect_model.names[cls_id]
                    track_id = track_ids[i]

                    cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
                    
                    label = f"ID:{track_id} CLS:{class_name} CONF:{conf}"
                    cv2.putText(frame, label, (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 0, 0), 2)

                    if class_name == "person":
                        for j, (ox1, oy1, ox2, oy2) in enumerate(xyxys):
                            if i == j:
                                continue
                            
                            other_class_name = detect_model.names[clss[j]]
                            if other_class_name == "person":
                                continue

                            xi1, yi1 = max(x1, ox1), max(y1, oy1)
                            xi2, yi2 = min(x2, ox2), min(y2, oy2)
                            
                            inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
                            if inter_area == 0:
                                continue
                                
                            obj_area = (ox2 - ox1) * (oy2 - oy1)
                            ioa = inter_area / obj_area if obj_area > 0 else 0.0

                            if ioa < 0.5:
                                continue

                            other_track_id = track_ids[j]
                            log_line += f"[IOA:{ioa:<7.4f} ID:{track_id:>04} CLS:{class_name:<8} ID:{other_track_id:>04} CLS:{other_class_name:<8}] "
                            ioa_detected = True

                            if kpts is not None:
                                in_box = (kpts[..., 0] >= ox1) & (kpts[..., 0] <= ox2) & (kpts[..., 1] >= oy1) & (kpts[..., 1] <= oy2)
                                if in_box.any():
                                    log_line2 += f"INTERACTING ID:{track_id:>04} CLS:{class_name:<8} ID:{other_track_id:>04} CLS:{other_class_name:<8} "
                                    interaction_detected = True

                writer.write(frame)
                if ioa_detected:
                    ioa_log.write(log_line + "\n")
                if interaction_detected:
                    hoi_log.write(log_line2 + "\n")

        except Exception as e:
            if writer is not None:
                writer.release()
            print(f"\n\nCrash detected!\n{e}")
            sys.exit(1)
        
        print(f"\nCompleted {video}\n")
        writer.release()

print("All Done")

Video: video\input\clip2.mp4
Processing Frame: 391/3921
Completed video\input\clip2.mp4

Video: video\input\clip4.mp4
Processing Frame: 68/693
Completed video\input\clip4.mp4

Video: video\input\clip5.mp4
Processing Frame: 218/2189
Completed video\input\clip5.mp4

All Done


In [18]:
# for video_index, video in enumerate(video_input_directory.glob("*.mp4")):
#     print(f"Video: {video}")

#     pose_results = pose_model.track(
#         device=0,
#         source=video,
#         tracker="botsort.yaml",
#         persist=True,
#         conf=0.5,
#         stream=True,
#         verbose=False,
#     )

#     cap = cv2.VideoCapture(str(video))
#     fps = int(cap.get(cv2.CAP_PROP_FPS))
#     width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1
#     cap.release()
    
#     output_video = str(video_output_directory / f"video{video_index}.mp4")
#     writer = cv2.VideoWriter(
#         output_video,
#         cv2.VideoWriter_fourcc(*"mp4v"),
#         fps,
#         (width, height)
#     )

#     try:
#         for result_index, result in enumerate(pose_results):
#             print(f"\rProcessing Frame: {result_index}/{total_frames}", end="", flush=True)
#             frame = result.plot()
#             writer.write(frame)

#     except Exception as e:
#         if 'writer' in locals() and writer is not None:
#             writer.release()
#         print(f"\n\nCrash detected!\n{e}")
#         sys.exit(1)
    
#     print(f"\nCompleted {video}\n")
#     writer.release()

# print("All Done")